# Task 2: Profile and Clean H2H Data

## Step 1: Profiling (discover issues)

In [2]:
import os
import json
import glob
import pandas as pd

RAW_DIR = "../data/raw"

# Find every H2H raw file fetched so far (works whether it's 90 or 153 files)
h2h_files = glob.glob(f"{RAW_DIR}/h2h_*.json")
h2h_files = [f for f in h2h_files if "progress" not in f]  # skip the progress tracker file

print(f"Found {len(h2h_files)} H2H raw files")

# Load and combine every fixture from every file into one flat list
all_fixtures = []
for filepath in h2h_files:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    all_fixtures.extend(data.get("response", []))

print(f"Total fixture records before dedup: {len(all_fixtures)}")

# Flatten the nested JSON into a table
df = pd.json_normalize(all_fixtures)

# Two team pairs can both include the same match once each is combined
# (e.g. h2h_2932_2938.json and a broader team's file might both contain
# the same Hilal-Ittihad fixture) - drop duplicates by fixture.id, the
# stable unique identifier from the API.
df = df.drop_duplicates(subset="fixture.id")

print(f"Total unique fixtures after dedup: {len(df)}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print()
print(df.head())

# Basic profiling: dtype, null count, percent null, unique count, sample value
profile_rows = []
for col in df.columns:
    profile_rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "null_count": df[col].isnull().sum(),
        "percent_null": round(df[col].isnull().mean() * 100, 1),
        "unique_count": df[col].nunique(),
        "sample_value": df[col].dropna().iloc[0] if df[col].notna().any() else None,
    })

profile_df = pd.DataFrame(profile_rows)
print("\nProfiling summary:")
print(profile_df.to_string(index=False))

Found 153 H2H raw files
Total fixture records before dedup: 1871
Total unique fixtures after dedup: 1871
Columns (40): ['fixture.id', 'fixture.referee', 'fixture.timezone', 'fixture.date', 'fixture.timestamp', 'fixture.periods.first', 'fixture.periods.second', 'fixture.venue.id', 'fixture.venue.name', 'fixture.venue.city', 'fixture.status.long', 'fixture.status.short', 'fixture.status.elapsed', 'fixture.status.extra', 'league.id', 'league.name', 'league.country', 'league.logo', 'league.flag', 'league.season', 'league.round', 'league.standings', 'teams.home.id', 'teams.home.name', 'teams.home.logo', 'teams.home.winner', 'teams.away.id', 'teams.away.name', 'teams.away.logo', 'teams.away.winner', 'goals.home', 'goals.away', 'score.halftime.home', 'score.halftime.away', 'score.fulltime.home', 'score.fulltime.away', 'score.extratime.home', 'score.extratime.away', 'score.penalty.home', 'score.penalty.away']

   fixture.id fixture.referee fixture.timezone               fixture.date  \
0      

## Step 2: Cleaning (apply fixes)

In [3]:
import re
import os
import pandas as pd

# Assumes `df` already exists (the combined, deduplicated raw dataframe
# from the profiling step earlier in this notebook).

df_clean = df.copy()

# --- 1. Fix data types ---
df_clean["fixture.date"] = pd.to_datetime(df_clean["fixture.date"])

# --- 2. Drop columns with no analytical value (logo/flag image URLs) ---
cols_to_drop = [c for c in df_clean.columns if "logo" in c or "flag" in c]

# --- 2b. Drop extratime/penalty columns entirely (team decision:
#     matches Abdulmajeed's cleaning approach for consistency) ---
cols_to_drop += [c for c in df_clean.columns if "extratime" in c or "penalty" in c]

df_clean = df_clean.drop(columns=cols_to_drop)

# --- 3. Standardize missing referee/venue as explicit "Unknown" marker
#     is NOT used here -- left as NaN, documented in the Issues List as
#     "missing at source" rather than invented. No action needed, this
#     is just a note for the README/Issues List.

# --- 4. Rename columns to snake_case ---
def to_snake_case(col: str) -> str:
    # "fixture.id" -> "fixture_id", "teams.home.name" -> "teams_home_name"
    col = col.replace(".", "_")
    col = re.sub(r"(?<!^)(?=[A-Z])", "_", col).lower()
    return col

df_clean.columns = [to_snake_case(c) for c in df_clean.columns]

# --- 5. Confirm no duplicates remain (should already be true from the
#     dedup step earlier, this is just a safety check) ---
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="fixture_id")
after = len(df_clean)
print(f"Duplicate check: {before - after} duplicate rows removed (expected 0)")

# --- 6. Save the deliverable (matches team naming: cleaned_fixtures.csv, cleaned_stats.csv) ---
os.makedirs("../data/interim", exist_ok=True)
df_clean.to_csv("../data/interim/cleaned_h2h.csv", index=False)

print(f"\nSaved cleaned_h2h.csv: {len(df_clean)} rows, {len(df_clean.columns)} columns")
print(f"\nColumns after cleaning:")
for c in df_clean.columns:
    print(f"  {c}")

df_clean.head()

Duplicate check: 0 duplicate rows removed (expected 0)

Saved cleaned_h2h.csv: 1871 rows, 32 columns

Columns after cleaning:
  fixture_id
  fixture_referee
  fixture_timezone
  fixture_date
  fixture_timestamp
  fixture_periods_first
  fixture_periods_second
  fixture_venue_id
  fixture_venue_name
  fixture_venue_city
  fixture_status_long
  fixture_status_short
  fixture_status_elapsed
  fixture_status_extra
  league_id
  league_name
  league_country
  league_season
  league_round
  league_standings
  teams_home_id
  teams_home_name
  teams_home_winner
  teams_away_id
  teams_away_name
  teams_away_winner
  goals_home
  goals_away
  score_halftime_home
  score_halftime_away
  score_fulltime_home
  score_fulltime_away


,fixture_id,fixture_referee,fixture_timezone,fixture_date,fixture_timestamp,fixture_periods_first,fixture_periods_second,fixture_venue_id,fixture_venue_name,fixture_venue_city,...,teams_home_winner,teams_away_id,teams_away_name,teams_away_winner,goals_home,goals_away,score_halftime_home,score_halftime_away,score_fulltime_home,score_fulltime_away
0,354407,None,UTC,2017-11-24 11:15:00+00:00,1511522100,1.511522e+09,1.511526e+09,NaN,None,None,...,True,10511,Al Riyadh,False,4.0,0.0,1.0,0.0,4.0,0.0
1,932039,None,UTC,2022-08-30 16:05:00+00:00,1661875500,1.661876e+09,1.661879e+09,NaN,Ar-Rass Stadium,Rass,...,True,10511,Al Riyadh,False,1.0,0.0,1.0,0.0,1.0,0.0
2,1252589,Sultan Alharbi,UTC,2025-02-08 13:55:00+00:00,1739022900,1.739023e+09,1.739026e+09,NaN,Ar-Rass Stadium,Rass,...,True,10511,Al Riyadh,False,3.0,2.0,0.0,1.0,3.0,2.0
3,1436119,M. Al Huwaish,UTC,2026-02-19 19:00:00+00:00,1771527600,1.771528e+09,1.771531e+09,NaN,Al-Hazem Club Stadium,Ar Rass,...,False,10511,Al Riyadh,True,0.0,2.0,0.0,2.0,0.0,2.0
4,1603186,None,UTC,2027-04-02 18:00:00+00:00,1806688800,NaN,NaN,NaN,Al-Hazem Club Stadium,Ar Rass,...,None,10511,Al Riyadh,None,NaN,NaN,NaN,NaN,NaN,NaN
